# CampusHireLens 🎓
## Placement Readiness & Skill-Gap Prediction using Machine Learning

**CampusHireLens** is an end-to-end beginner-friendly machine learning project designed to estimate a student's placement-readiness level from academic performance, coding practice, projects, internships, communication practice, aptitude performance, and interview preparation.

### What makes this project different?
Instead of only predicting a binary outcome, the project combines **ML prediction + explainable skill-gap recommendations**. It can tell a student whether their readiness is Low, Medium, or High and identify practical areas to improve.

**Tools:** Python, Pandas, NumPy, Matplotlib, scikit-learn


## 1. Problem Statement

Students often know their marks but do not know which employability skills need improvement before placements. This project builds a machine-learning workflow that predicts a student's readiness category and produces a simple personalized improvement plan.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

np.random.seed(42)


## 2. Build a self-contained learning dataset

The dataset is synthetically generated so the notebook can run without downloading private or copyrighted student data. The features represent realistic placement-preparation activities.

In [ ]:
n = 1200
df = pd.DataFrame({
    'cgpa': np.round(np.random.uniform(5.5, 10.0, n), 2),
    'coding_hours_week': np.random.randint(0, 16, n),
    'dsa_problems': np.random.randint(0, 301, n),
    'projects': np.random.randint(0, 5, n),
    'internship_months': np.random.randint(0, 13, n),
    'aptitude_score': np.random.randint(30, 101, n),
    'communication_score': np.random.randint(30, 101, n),
    'mock_interviews': np.random.randint(0, 11, n),
})

# Create a continuous readiness score with a small amount of noise.
score = (
    10 * df['cgpa'] +
    1.5 * df['coding_hours_week'] +
    0.06 * df['dsa_problems'] +
    7 * df['projects'] +
    1.8 * df['internship_months'] +
    0.35 * df['aptitude_score'] +
    0.30 * df['communication_score'] +
    2.5 * df['mock_interviews'] +
    np.random.normal(0, 12, n)
)

df['readiness'] = pd.cut(
    score,
    bins=[-np.inf, 125, 165, np.inf],
    labels=['Low', 'Medium', 'High']
)

df.head()

In [ ]:
print('Shape:', df.shape)
print('\nMissing values:')
print(df.isnull().sum())
print('\nReadiness distribution:')
print(df['readiness'].value_counts())

## 3. Exploratory Data Analysis

In [ ]:
df['readiness'].value_counts().plot(kind='bar')
plt.title('Placement Readiness Distribution')
plt.xlabel('Readiness Level')
plt.ylabel('Number of Students')
plt.xticks(rotation=0)
plt.show()

In [ ]:
df.groupby('readiness')['projects'].mean().reindex(['Low','Medium','High']).plot(kind='bar')
plt.title('Average Projects by Readiness Level')
plt.xlabel('Readiness Level')
plt.ylabel('Average Number of Projects')
plt.xticks(rotation=0)
plt.show()

In [ ]:
features = ['cgpa','coding_hours_week','dsa_problems','projects','internship_months','aptitude_score','communication_score','mock_interviews']
print(df[features + ['readiness']].groupby('readiness')[features].mean().round(2))

## 4. Train the Machine Learning Model

In [ ]:
X = df[features]
y = df['readiness']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = RandomForestClassifier(
    n_estimators=250,
    max_depth=8,
    random_state=42
)
model.fit(X_train_scaled, y_train)
pred = model.predict(X_test_scaled)


## 5. Evaluate the Model

In [ ]:
print('Test Accuracy:', round(accuracy_score(y_test, pred), 3))
print('\nClassification Report:\n')
print(classification_report(y_test, pred))

cm = confusion_matrix(y_test, pred, labels=['Low','Medium','High'])
print('Confusion Matrix (Low, Medium, High):\n', cm)

## 6. Understand What the Model Uses

Feature importance gives an interpretable view of which inputs contributed most to the model's decisions.

In [ ]:
importance = pd.Series(model.feature_importances_, index=features).sort_values(ascending=True)
importance.plot(kind='barh')
plt.title('Random Forest Feature Importance')
plt.xlabel('Importance')
plt.show()

print('Most influential features:')
print(importance.sort_values(ascending=False).head(5).round(3))

## 7. Personalized Skill-Gap Analyzer

The ML model predicts the readiness category. A rule-based layer then turns low individual scores into actionable suggestions. This makes the project more useful than a prediction-only notebook.

In [ ]:
def placement_advice(student):
    row = pd.DataFrame([student])[features]
    row_scaled = scaler.transform(row)
    prediction = model.predict(row_scaled)[0]
    advice = []

    if student['coding_hours_week'] < 5:
        advice.append('Increase coding practice to at least 5 hours/week.')
    if student['dsa_problems'] < 100:
        advice.append('Practice more DSA problems, focusing on arrays, strings, trees and graphs.')
    if student['projects'] < 2:
        advice.append('Build at least one complete project and document it on GitHub.')
    if student['internship_months'] == 0:
        advice.append('Seek an internship, virtual internship, or practical industry project.')
    if student['aptitude_score'] < 70:
        advice.append('Practice quantitative aptitude, logical reasoning and timed tests.')
    if student['communication_score'] < 70:
        advice.append('Practice technical explanations, HR questions and mock introductions.')
    if student['mock_interviews'] < 3:
        advice.append('Complete at least three mock interviews and review mistakes.')
    if student['cgpa'] < 7:
        advice.append('Maintain/improve academic performance where possible.')

    return prediction, advice

sample_student = {
    'cgpa': 7.8,
    'coding_hours_week': 6,
    'dsa_problems': 120,
    'projects': 2,
    'internship_months': 3,
    'aptitude_score': 76,
    'communication_score': 72,
    'mock_interviews': 4
}

level, advice = placement_advice(sample_student)
print('Predicted readiness:', level)
print('\nPersonalized improvement plan:')
for item in advice:
    print('-', item)

## 8. Conclusion

CampusHireLens demonstrates a complete machine-learning workflow: **problem definition → dataset creation → EDA → preprocessing → model training → evaluation → feature interpretation → personalized recommendations**.

### Future Enhancements
- Replace synthetic data with a real, ethically collected placement dataset.
- Compare Random Forest with Logistic Regression, XGBoost/Gradient Boosting, and other models.
- Add cross-validation and hyperparameter tuning.
- Build a Streamlit dashboard where students enter their details and receive readiness and skill-gap feedback.
- Add explainable-AI techniques such as SHAP for individual predictions.

> **Note:** This notebook is an educational prototype. The generated dataset and prediction should not be treated as a real hiring or placement decision system.